# Predictive performance across OpenML-CC18

**Hypothesis:** Forest Sketch with one, two, or three iterations produces a distinguishable held-out accuracy relative to a standard random forest, and the difference is consistent across a multi-dataset OpenML-CC18 sample.

Each `(dataset_id, repetition_id)` pair is one paired block. The critical-difference diagram therefore ranks standard random forest, `T=1`, `T=2`, and `T=3` within every dataset/repetition block. The joint-block analysis is exploratory; a confirmatory analysis should also aggregate repetitions within each dataset so that repeated seeds do not artificially increase the apparent number of independent datasets.

In [9]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.exceptions import DataDimensionalityWarning

warnings.filterwarnings('ignore', category=DataDimensionalityWarning)

from _hypothesis_utils import (
    DEFAULT_DIMENSION_RATIO,
    DEFAULT_N_ESTIMATORS,
    classification_score,
    forest_classifier,
    make_datasets,
    openml_classification_split,
    openml_downstream_classifier,
    forest_sketch,
    plot_critical_difference,
    timed_fit_transform,
)

DATASET_COUNT = 4
MAX_DATASET_SIZE = 1000
DATASETS = make_datasets(n=DATASET_COUNT, max_size=MAX_DATASET_SIZE, seed=42)
REPETITIONS = tuple(range(5))
ITERATIONS = (1, 2, 3)
METHODS = ('Random Forest', 'Forest Sketch T=1', 'Forest Sketch T=2', 'Forest Sketch T=3')
N_ESTIMATORS = 100
DIMENSION_RATIO = 20

dataset_summary = pd.DataFrame([
    {
        'dataset_id': dataset.dataset_id,
        'name': dataset.name,
        'n_original': dataset.n_original,
        'n_used': dataset.n_used,
        'n_features': len(dataset.feature_names),
    }
    for dataset in DATASETS
])
display(dataset_summary)

,dataset_id,name,n_original,n_used,n_features
0,3,kr-vs-kp,3196,1000,36
1,6,letter,20000,1000,16
2,11,balance-scale,625,625,4
3,12,mfeat-factors,2000,1000,216


In [ ]:
rows = []
for dataset in DATASETS:
    for repetition_id in REPETITIONS:
        X_train, X_test, y_train, y_test, preprocessor = openml_classification_split(
            dataset, seed=repetition_id
        )
        p = X_train.shape[1]
        # The estimator resolves d=DIMENSION_RATIO*p from the encoded width.

        # Standard random forest on the encoded input is the reference model.
        rf_accuracy, rf_errors = classification_score(
            forest_classifier(repetition_id, n_estimators=N_ESTIMATORS),
            X_train, y_train, X_test, y_test,
        )
        for iterations in ITERATIONS:
            sketch = forest_sketch(
                repetition_id,
                dimension_ratio=DIMENSION_RATIO,
                n_iterations=iterations,
                n_estimators=N_ESTIMATORS,
            )
            X_train_view, fit_wall, fit_cpu = timed_fit_transform(
                sketch, X_train, y_train
            )
            dimension = sketch.n_components_
            X_test_view = sketch.transform(X_test)
            accuracy, errors = classification_score(
                openml_downstream_classifier(repetition_id),
                X_train_view, y_train, X_test_view, y_test,
            )
            rows.append({
                'task_id': dataset.task_id,
                'dataset_id': dataset.dataset_id,
                'dataset': dataset.name,
                'repetition_id': repetition_id,
                'n_original': dataset.n_original,
                'n_used': dataset.n_used,
                'p_encoded': p,
                'dimension': dimension,
                'iterations': iterations,
                'method': f'Forest Sketch T={iterations}',
                'accuracy': accuracy,
                'errors': errors,
                'fit_wall_seconds': fit_wall,
                'fit_cpu_seconds': fit_cpu,
            })

        rows.append({
            'task_id': dataset.task_id,
            'dataset_id': dataset.dataset_id,
            'dataset': dataset.name,
            'repetition_id': repetition_id,
            'n_original': dataset.n_original,
            'n_used': dataset.n_used,
            'p_encoded': p,
            'dimension': dimension,
            'iterations': 0,
            'method': 'Random Forest',
            'accuracy': rf_accuracy,
            'errors': rf_errors,
        })

results = pd.DataFrame(rows)
results.sort_values(['dataset_id', 'repetition_id', 'iterations'])

In [ ]:
summary = results.groupby(['dataset', 'method'], as_index=False).agg(
    mean_accuracy=('accuracy', 'mean'),
    std_accuracy=('accuracy', 'std'),
)
display(summary)

ax = summary.pivot(index='dataset', columns='method', values='mean_accuracy').plot(
    kind='bar', figsize=(8, 4),
)
y_values = summary['mean_accuracy'].to_numpy()
y_padding = max(0.01, 0.05 * (y_values.max() - y_values.min()))
ax.set_ylim(y_values.min() - y_padding, y_values.max() + y_padding)
ax.set_ylabel('Mean held-out accuracy')
ax.set_xlabel('')
ax.set_title('OpenML-CC18 predictive performance')
ax.legend(title='Method', bbox_to_anchor=(1.04, 1), loc='upper left')


In [ ]:
plot_critical_difference(
    results,
    block_column=['dataset_id', 'repetition_id'],
    method_column='method',
    score_column='accuracy',
    title='Critical differences over dataset × repetition blocks',
)

paired = results.pivot(
    index=['dataset_id', 'repetition_id'],
    columns='method',
    values='accuracy',
)
reference = paired['Random Forest']
overall_gain_rows = []
per_dataset_gain_rows = []
for method in METHODS[1:]:
    gain = paired[method] - reference
    half_width = t.ppf(0.975, len(gain) - 1) * gain.std(ddof=1) / np.sqrt(len(gain))
    overall_gain_rows.append({
        'method': method,
        'reference': 'Random Forest',
        'mean_gain': gain.mean(),
        'ci_low': gain.mean() - half_width,
        'ci_high': gain.mean() + half_width,
        'n_blocks': len(gain),
    })
    dataset_stats = gain.groupby(level=0).agg(['mean', 'std'])
    for dataset_id, stats in dataset_stats.iterrows():
        per_dataset_gain_rows.append({
            'dataset_id': dataset_id,
            'method': method,
            'mean_gain': stats['mean'],
            'std_gain': stats['std'],
        })

overall_gain_summary = pd.DataFrame(overall_gain_rows)
per_dataset_gain_summary = pd.DataFrame(per_dataset_gain_rows)
display(overall_gain_summary.style.format({
    'mean_gain': '{:+.3f}',
    'ci_low': '{:+.3f}',
    'ci_high': '{:+.3f}',
}))
display(per_dataset_gain_summary.style.format({
    'mean_gain': '{:+.3f}',
    'std_gain': '{:.3f}',
}))

The primary evidence is the CD diagram and the paired gain of each Forest Sketch iteration relative to the standard random forest. A positive overall gain is not sufficient for a cross-dataset claim if it is driven by one dataset; the per-dataset paired gains should be reported as well.